# Fight Night Fan Intelligence - Cleaning Notebook
Sample (synthetic) data. Run each cell in order with Shift+Enter.

### CELL 1 - unzip the data kit you uploaded (Files panel, upload fight-night-fan-intelligence.zip)

In [5]:
import os, zipfile
zip_path = "/content/fight night/Fight Night Fan Intelligence Data Kit (1).zip"
if os.path.exists(zip_path):
    zipfile.ZipFile(zip_path).extractall(".")
print(os.listdir("data_raw"))

['partner_series_funnel_raw.csv', 'cities_lookup.csv', 'fx_rates.csv', 'ticket_orders_raw.csv', 'subscribers_raw.csv', 'viewership_raw.csv', 'events.csv']


### CELL 2 - imports and folders

In [6]:
import pandas as pd, numpy as np
R, O = "data_raw", "data_clean"
os.makedirs(O, exist_ok=True)
pd.set_option("display.max_columns", 50)

### CELL 3 - load every raw file

In [7]:
tickets = pd.read_csv(f"{R}/ticket_orders_raw.csv", keep_default_na=False)
subs    = pd.read_csv(f"{R}/subscribers_raw.csv", keep_default_na=False)
view    = pd.read_csv(f"{R}/viewership_raw.csv")
events  = pd.read_csv(f"{R}/events.csv")
cities  = pd.read_csv(f"{R}/cities_lookup.csv")
fx      = pd.read_csv(f"{R}/fx_rates.csv")
partner = pd.read_csv(f"{R}/partner_series_funnel_raw.csv")
for name, df in [("tickets",tickets),("subs",subs),("view",view),("events",events),("partner",partner)]:
    print(name, df.shape)

tickets (54083, 11)
subs (25375, 12)
view (480, 8)
events (48, 22)
partner (240, 11)


### CELL 4 - profile the tickets file (look before you clean)

In [8]:
tickets.info()
print(tickets.Seat_Tier.value_counts())
print(tickets.Currency.value_counts())
print("duplicate Order_IDs:", tickets.Order_ID.duplicated().sum())
print("blank home cities:", (tickets.Fan_Home_City == "").sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54083 entries, 0 to 54082
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order_ID                54083 non-null  object 
 1   Event_ID                54083 non-null  object 
 2   Fan_ID                  54083 non-null  object 
 3   Email_Hash              54083 non-null  object 
 4   Purchase_Date           54083 non-null  object 
 5   Seat_Tier               54083 non-null  object 
 6   Quantity                54083 non-null  int64  
 7   Price_Per_Ticket_Local  54083 non-null  float64
 8   Currency                54083 non-null  object 
 9   Fan_Home_City           54083 non-null  object 
 10  Sales_Channel           54083 non-null  object 
dtypes: float64(1), int64(1), object(9)
memory usage: 4.5+ MB
Seat_Tier
Upper Bowl     15310
Lower Bowl     14562
Floor           8504
VIP Package     3197
Upper           1942
UPPER BOWL      1854
lowe

### CELL 5 - remove duplicate orders (expect 1,060)

In [9]:
before = len(tickets)
tickets = tickets.drop_duplicates(subset="Order_ID")
print("removed", before - len(tickets))

removed 1060


### CELL 6 - fix mixed date formats

In [10]:
tickets["Purchase_Date"] = pd.to_datetime(tickets.Purchase_Date, format="mixed")
print(tickets.Purchase_Date.min(), tickets.Purchase_Date.max())

2024-06-19 00:00:00 2026-09-26 00:00:00


### CELL 7 - standardize seat tiers to 5 values

In [11]:
tier_map = {"ringside":"Ringside","ring side":"Ringside","vip package":"VIP Package","vip pkg":"VIP Package","vip":"VIP Package",
            "floor":"Floor","floor seats":"Floor","lower bowl":"Lower Bowl","lower":"Lower Bowl",
            "upper bowl":"Upper Bowl","upper":"Upper Bowl"}
tickets["Seat_Tier"] = tickets.Seat_Tier.str.strip().str.lower().map(tier_map)
print(tickets.Seat_Tier.value_counts(dropna=False))   # no NaN = every variant was mapped

Seat_Tier
Upper Bowl     18714
Lower Bowl     17791
Floor          10403
VIP Package     3898
Ringside        2217
Name: count, dtype: int64


### CELL 8 - standardize home cities

In [12]:
city_map = {"nyc":"New York","new york city":"New York","la":"Los Angeles","l.a.":"Los Angeles",
            "vegas":"Las Vegas","cdmx":"Mexico City","ciudad de mexico":"Mexico City"}
c = tickets.Fan_Home_City.str.strip()
tickets["Fan_Home_City"] = c.str.lower().map(city_map).fillna(c.str.title())
print("blanks:", (tickets.Fan_Home_City == "").sum())          # expect 795
tickets["Fan_Home_City"] = tickets.Fan_Home_City.replace("", "Unknown")
print(sorted(tickets.Fan_Home_City.unique()))

blanks: 795
['Atlanta', 'Belfast', 'Berlin', 'Birmingham', 'Cardiff', 'Chicago', 'Dublin', 'Glasgow', 'Guadalajara', 'Houston', 'Las Vegas', 'London', 'Los Angeles', 'Madrid', 'Manchester', 'Melbourne', 'Mexico City', 'Miami', 'Monterrey', 'New York', 'Newark', 'Philadelphia', 'Phoenix', 'Riyadh', 'San Antonio', 'Sydney', 'Tokyo', 'Toronto', 'Unknown']


### CELL 9 - convert every price to USD and flag comps

In [13]:
tickets["Price_Per_Ticket_Local"] = tickets.Price_Per_Ticket_Local.astype(float)
tickets = tickets.merge(fx, on="Currency", how="left")
tickets["Price_Per_Ticket_USD"] = (tickets.Price_Per_Ticket_Local * tickets.USD_per_unit).round(2)
tickets["Is_Comp"] = tickets.Price_Per_Ticket_USD == 0
tickets["Order_Revenue_USD"] = tickets.Price_Per_Ticket_USD * tickets.Quantity
print("comps:", tickets.Is_Comp.sum())                           # expect 212

comps: 212


### CELL 10 - add event info and fan home coordinates

In [14]:
tickets = tickets.merge(events[["Event_ID","Event_City","Event_Country","Event_Date"]], on="Event_ID", how="left")
home = cities.rename(columns={"City":"Fan_Home_City","Country":"Fan_Home_Country","Region":"Fan_Home_Region",
                              "Latitude":"Home_Lat","Longitude":"Home_Lon"})
tickets = tickets.merge(home, on="Fan_Home_City", how="left")
ev_xy = cities.rename(columns={"City":"Event_City","Latitude":"Ev_Lat","Longitude":"Ev_Lon"})[["Event_City","Ev_Lat","Ev_Lon"]]
tickets = tickets.merge(ev_xy, on="Event_City", how="left")

### CELL 11 - distance traveled (haversine) and traveler flags

In [15]:
p = np.pi / 180
a = (np.sin((tickets.Ev_Lat - tickets.Home_Lat) * p / 2) ** 2
     + np.cos(tickets.Home_Lat * p) * np.cos(tickets.Ev_Lat * p) * np.sin((tickets.Ev_Lon - tickets.Home_Lon) * p / 2) ** 2)
tickets["Distance_Km"] = (12742 * np.arcsin(np.sqrt(a))).round(0)
tickets["Traveler"] = np.where(tickets.Distance_Km.isna(), "Unknown",
                       np.where(tickets.Distance_Km > 160, "Traveler (100+ mi)", "Local"))
tickets["International_Traveler"] = tickets.Fan_Home_Country.notna() & (tickets.Fan_Home_Country != tickets.Event_Country)
tickets["Days_Before_Event"] = (pd.to_datetime(tickets.Event_Date) - tickets.Purchase_Date).dt.days
print(tickets.Traveler.value_counts(normalize=True).round(3))

Traveler
Traveler (100+ mi)    0.525
Local                 0.460
Unknown               0.015
Name: proportion, dtype: float64


### CELL 12 - save clean tickets

In [16]:
tickets.drop(columns=["USD_per_unit","Ev_Lat","Ev_Lon"]).to_csv(f"{O}/ticket_orders_clean.csv", index=False)
print("tickets saved:", len(tickets))

tickets saved: 53023


### CELL 13 - clean subscribers

In [17]:
before = len(subs); subs = subs.drop_duplicates(subset="Account_ID"); print("removed", before - len(subs))   # expect 375
plan_map = {"standard monthly":"Standard Monthly","standard-monthly":"Standard Monthly","std monthly":"Standard Monthly",
            "standard annual":"Standard Annual","annual":"Standard Annual","ultimate annual":"Ultimate Annual",
            "ultimate":"Ultimate Annual","ppv only":"PPV Only","ppv":"PPV Only","ppv-only":"PPV Only"}
subs["Plan"] = subs.Plan.str.strip().str.lower().map(plan_map)
iso = {"US":"United States","CA":"Canada","MX":"Mexico","GB":"United Kingdom","IE":"Ireland","ES":"Spain",
       "DE":"Germany","SA":"Saudi Arabia","AU":"Australia","JP":"Japan"}
subs["Country"] = subs.Country.replace(iso).str.title()
subs["Signup_Date"] = pd.to_datetime(subs.Signup_Date, format="mixed")
subs["Cancel_Date"] = pd.to_datetime(subs.Cancel_Date.replace("", None))
subs["Churned"] = subs.Cancel_Date.notna()
subs["Days_To_Cancel"] = (subs.Cancel_Date - subs.Signup_Date).dt.days
subs["Churned_60d"] = subs.Days_To_Cancel.le(60)
subs["Signup_Type"] = np.where(subs.Signup_Trigger_Event != "", "Fight-triggered", "Always-on")
subs["Is_Ticket_Buyer"] = subs.Email_Hash.isin(set(tickets.Email_Hash))
print(subs.Plan.value_counts(dropna=False)); print(subs.Country.value_counts())
subs.to_csv(f"{O}/subscribers_clean.csv", index=False)

removed 375
Plan
Standard Monthly    11182
Standard Annual      5031
PPV Only             4996
Ultimate Annual      3791
Name: count, dtype: int64
Country
United States     11629
United Kingdom     6012
Mexico             2843
Ireland            1153
Australia           798
Saudi Arabia        786
Canada              621
Japan               610
Spain               288
Germany             260
Name: count, dtype: int64


### CELL 14 - clean viewership + local start hour

In [18]:
view["Country"] = view.Country.replace(iso)
print("blank signups:", view.New_Signups.isna().sum())          # expect 14, leave blank
view = view.merge(events[["Event_ID","Broadcast","Main_Event_Start_ET","Combined_Star_Power","Event_Country",
                          "Headliner_A_Country","Headliner_B_Country"]], on="Event_ID", how="left")
offset = {"United States":0,"Canada":0,"Mexico":-1,"United Kingdom":5,"Ireland":5,"Spain":6,"Germany":6,
          "Saudi Arabia":7,"Australia":15,"Japan":13}
view["Local_Main_Event_Hour"] = (view.Main_Event_Start_ET + view.Country.map(offset)) % 24
view["Overnight_Slot"] = view.Local_Main_Event_Hour.between(1, 6.99)
view.to_csv(f"{O}/viewership_clean.csv", index=False)

blank signups: 14


### CELL 15 - clean the partner cable series funnel

In [19]:
mk = {"NYC":"New York","New York, NY":"New York","LA":"Los Angeles","Los Angeles, CA":"Los Angeles",
      "Washington, D.C.":"Washington DC","DC":"Washington DC","Dallas-Ft. Worth":"Dallas","DFW":"Dallas"}
partner["Market"] = partner.Market.replace(mk)
partner["TV_Viewers"] = partner.TV_Viewers.astype(str).str.replace(",", "").astype(int)
partner["Signup_Rate"] = partner.Promo_Code_Signups / partner.TV_Viewers
partner["Retention_30d"] = partner.Retained_30d / partner.Promo_Code_Signups
partner["Retention_60d"] = partner.Retained_60d / partner.Promo_Code_Signups
print("markets:", partner.Market.nunique())                     # expect 20
partner.to_csv(f"{O}/partner_series_funnel_clean.csv", index=False)

markets: 20


### CELL 16 - fan profiles (one row per ticket buyer) + segments

In [20]:
fp = tickets.groupby(["Fan_ID","Email_Hash","Fan_Home_City","Fan_Home_Country"], dropna=False).agg(
    Events_Attended=("Event_ID","nunique"), Revenue_USD=("Order_Revenue_USD","sum"),
    Premium_Orders=("Seat_Tier", lambda x: x.isin(["Ringside","VIP Package"]).sum()),
    Travel_Events=("Traveler", lambda x: (x == "Traveler (100+ mi)").sum())).reset_index()
fp["Is_DAZN_Subscriber"] = fp.Email_Hash.isin(set(subs.Email_Hash))
fp["Segment"] = np.select([fp.Events_Attended >= 3, fp.Premium_Orders > 0, fp.Travel_Events > 0],
                          ["Superfan (3+ fights)", "Premium buyer", "Traveler"], "Casual")
fp.to_csv(f"{O}/fan_profiles.csv", index=False)
print(fp.Segment.value_counts()); print("not subscribed:", round(1 - fp.Is_DAZN_Subscriber.mean(), 3))

Segment
Traveler                16039
Casual                  14945
Premium buyer            4911
Superfan (3+ fights)     1795
Name: count, dtype: int64
not subscribed: 0.62


### CELL 17 - city opportunity table

In [21]:
dem = fp[fp.Fan_Home_City != "Unknown"].groupby("Fan_Home_City").agg(Ticket_Fans=("Fan_ID","count"),
        Traveling_Fans=("Travel_Events", lambda x: (x > 0).sum()), Ticket_Revenue_USD=("Revenue_USD","sum"))
sb = subs.groupby("City").agg(Subscribers=("Account_ID","count"))
eh = events.groupby("Event_City").size().rename("Events_Hosted")
opp = cities.set_index("City").join(dem).join(sb).join(eh).fillna({"Events_Hosted": 0})
opp["Opportunity_Score"] = ((opp.Subscribers + opp.Traveling_Fans) / (opp.Events_Hosted + 1)).round(0)
opp.reset_index().to_csv(f"{O}/city_opportunity.csv", index=False)
print(opp.sort_values("Opportunity_Score", ascending=False).head(5)[["Events_Hosted","Opportunity_Score"]])

             Events_Hosted  Opportunity_Score
City                                         
Houston                0.0             4093.0
Mexico City            0.0             3758.0
Chicago                0.0             3728.0
Miami                  0.0             1466.0
Phoenix                0.0             1441.0


### CELL 18 - events with coordinates + copy lookup

In [22]:
events.merge(cities[["City","Latitude","Longitude"]].rename(columns={"City":"Event_City"}), on="Event_City",
             how="left").to_csv(f"{O}/events_clean.csv", index=False)
cities.to_csv(f"{O}/cities_lookup.csv", index=False)

### CELL 19 - QA checks (every line should print True)

In [23]:
print(tickets.Order_ID.is_unique, subs.Account_ID.is_unique, tickets.Seat_Tier.notna().all(),
      subs.Plan.notna().all(), tickets.Price_Per_Ticket_USD.notna().all(), view.Country.nunique() == 10)

True True True True True True


### CELL 20 - download the clean folder

In [24]:
import shutil; shutil.make_archive("data_clean", "zip", O)
try:
    from google.colab import files; files.download("data_clean.zip")
except ImportError:
    print("Not in Colab - data_clean.zip saved locally")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>